In [7]:
%pip install -q -U weasyprint

Note: you may need to restart the kernel to use updated packages.


In [103]:
from pathlib import Path
import json
import re
import html as html_lib

import fitz
from bs4 import BeautifulSoup
from weasyprint import HTML

In [468]:
ROOT = Path.home() / "gemini HTML output"

LANGUAGE = "Assamese"
DOC_NAME = "Lentil_KVK , Thoubal ,Manipur"

WORKDIR_ROOT = ROOT / "Workdir" / LANGUAGE / DOC_NAME

START_PAGE = 1
END_PAGE = 2

OVERWRITE_EXISTING = True

PDF_RENDER_MAX_RETRIES = 3
MIN_TEXT_MATCH_RATIO = 0.55   

print("Workdir root:", WORKDIR_ROOT)
print("Exists:", WORKDIR_ROOT.exists())
print("Start page:", START_PAGE)
print("End page:", END_PAGE)

Workdir root: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur
Exists: True
Start page: 1
End page: 2


In [469]:
def build_print_html(fragment: str, title: str) -> str:
    return f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>{title}</title>
  <style>
    @page {{
      size: A4;
      margin: 18mm;
    }}

    body {{
      font-family:
        "Noto Sans",
        "Noto Sans Devanagari",
        "DejaVu Sans",
        "Liberation Sans",
        Arial,
        sans-serif;
      font-size: 11pt;
      line-height: 1.4;
      color: #111;
    }}

    h1, h2, h3, h4, h5, h6 {{
      margin-top: 14px;
      margin-bottom: 8px;
    }}

    p {{
      margin: 0 0 10px 0;
    }}

    table {{
      border-collapse: collapse;
      width: 100%;
      table-layout: fixed;
      margin: 8px 0 12px 0;
    }}

    th, td {{
      border: 1px solid #444;
      padding: 6px;
      vertical-align: top;
      word-wrap: break-word;
      overflow-wrap: anywhere;
    }}

    figure {{
      margin: 14px 0;
      text-align: center;
      page-break-inside: avoid;
      break-inside: avoid;
    }}

    img {{
      max-width: 100%;
      height: auto;
      display: block;
      margin: 0 auto;
    }}

    figcaption {{
      margin-top: 8px;
      font-size: 10pt;
    }}

    ul, ol {{
      margin-top: 6px;
      margin-bottom: 10px;
    }}
  </style>
</head>
<body>
{fragment}
</body>
</html>
"""

In [470]:
def extract_expected_text_from_html(html_path: Path) -> str:
    html_text = html_path.read_text(encoding="utf-8")
    soup = BeautifulSoup(html_text, "html.parser")

    # remove non-visible/script-like nodes if present
    for tag in soup(["script", "style"]):
        tag.decompose()

    text = soup.get_text(separator=" ", strip=True)
    text = html_lib.unescape(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [471]:
def extract_text_from_pdf(pdf_path: Path) -> str:
    doc = fitz.open(pdf_path)
    parts = []
    for page in doc:
        parts.append(page.get_text("text"))
    doc.close()

    text = " ".join(parts)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [472]:
def validate_generated_pdf(html_path: Path, pdf_path: Path, min_match_ratio: float = 0.55):
    expected = extract_expected_text_from_html(html_path)
    actual = extract_text_from_pdf(pdf_path)

    if not expected:
        return {
            "ok": False,
            "reason": "expected_html_text_empty",
            "expected_len": 0,
            "actual_len": len(actual),
            "match_ratio": 0.0,
        }

    expected_tokens = expected.split()
    actual_tokens = set(actual.split())

    matched = sum(1 for tok in expected_tokens if tok in actual_tokens)
    match_ratio = matched / max(len(expected_tokens), 1)

    ok = len(actual.strip()) > 0 and match_ratio >= min_match_ratio

    return {
        "ok": ok,
        "reason": "ok" if ok else "text_mismatch",
        "expected_len": len(expected),
        "actual_len": len(actual),
        "match_ratio": match_ratio,
    }

In [473]:
def convert_final_html_to_pdf_with_retry(page_dir: Path):
    final_html_path = page_dir / "final_with_images.html"
    preview_html_path = page_dir / "final_with_images_preview.html"
    final_pdf_path = page_dir / "final_page.pdf"

    if not final_html_path.exists():
        raise FileNotFoundError(f"final_with_images.html not found: {final_html_path}")

    fragment = final_html_path.read_text(encoding="utf-8")
    full_html = build_print_html(fragment, title=page_dir.name)
    preview_html_path.write_text(full_html, encoding="utf-8")

    last_validation = None

    for attempt in range(1, PDF_RENDER_MAX_RETRIES + 1):
        HTML(
            filename=str(preview_html_path),
            base_url=str(page_dir),
        ).write_pdf(str(final_pdf_path))

        validation = validate_generated_pdf(
            html_path=final_html_path,
            pdf_path=final_pdf_path,
            min_match_ratio=MIN_TEXT_MATCH_RATIO,
        )

        print(f"{page_dir.name} | attempt {attempt} | validation: {validation}")

        if validation["ok"]:
            return {
                "preview_html": preview_html_path,
                "final_pdf": final_pdf_path,
                "validation": validation,
                "attempts_used": attempt,
            }

        last_validation = validation

    raise RuntimeError(
        f"PDF validation failed after {PDF_RENDER_MAX_RETRIES} attempts for {page_dir.name}. "
        f"Last validation: {last_validation}"
    )

In [474]:
summary = []

for page_num in range(START_PAGE, END_PAGE + 1):
    page_name = f"page_{page_num:03d}"
    page_dir = WORKDIR_ROOT / page_name

    final_html_path = page_dir / "final_with_images.html"
    final_pdf_path = page_dir / "final_page.pdf"
    err_path = page_dir / "final_pdf_error.txt"

    if not final_html_path.exists():
        msg = f"final_with_images.html not found: {final_html_path}"
        print(msg)
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": msg,
        })
        continue

    if final_pdf_path.exists() and not OVERWRITE_EXISTING:
        print(f"Skipping {page_name}: final_page.pdf already exists")
        summary.append({
            "page": page_name,
            "status": "skipped",
            "reason": "final_page.pdf already exists",
        })
        continue

    print(f"\n=== Processing {page_name} ===")

    try:
        result = convert_final_html_to_pdf_with_retry(page_dir)

        summary.append({
            "page": page_name,
            "status": "success",
            "preview_html": str(result["preview_html"]),
            "final_pdf": str(result["final_pdf"]),
            "validation": result["validation"],
            "attempts_used": result["attempts_used"],
        })
        print(f"Saved: {result['final_pdf']}")

    except Exception as e:
        err = str(e)
        err_path.write_text(err, encoding="utf-8")
        summary.append({
            "page": page_name,
            "status": "failed",
            "error": err,
            "error_file": str(err_path),
        })
        print(f"Failed: {page_name} -> {err}")


=== Processing page_001 ===
page_001 | attempt 1 | validation: {'ok': True, 'reason': 'ok', 'expected_len': 908, 'actual_len': 908, 'match_ratio': 1.0}
Saved: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_001/final_page.pdf

=== Processing page_002 ===
page_002 | attempt 1 | validation: {'ok': True, 'reason': 'ok', 'expected_len': 2813, 'actual_len': 2826, 'match_ratio': 1.0}
Saved: /home/dhruva/gemini HTML output/Workdir/Assamese/Lentil_KVK , Thoubal ,Manipur/page_002/final_page.pdf
